# Example: STED-FM — Synaptic protein phenotypes

Compare a foundation model trained on STED microscopy with DINOv2 on super-resolution images of synaptic proteins.

**Datasets:** downloaded and reshaped by the helpers in [`sted_fm/`](sted_fm/):

- **NAS:** 224×224 crops of PSD-95 under four activity conditions (0MgGlyBic, 48hTTX, Block, GluGly).
- **Optim:** 224×224 crops of four proteins (CaMKII, PSD-95, actin, tubulin).
- **Zooniverse:** crops of individual synaptic protein clusters. This is the only set with masks suited to classical properties.

**Key steps:**
1. Install STED-FM and turn the chosen dataset into PhenoMe's folder layout.
2. Wrap the model. STED-FM expects one channel and does not use ImageNet normalization.
3. Extract embeddings, compute the `complete` property preset, and explore the projection.

To use your own images instead, skip the download and point `images_dir`, `masks_dir`, and `metadata_dir` at a folder with `images/`, optional `masks/`, and `metadata.csv`.


## 0. Install STED-FM

STED-FM is not a PhenoMe dependency. Install it in this environment before the cells below:

```bash
pip install "git+https://github.com/FLClab/STED-FM.git"
pip install requests
```

`requests` is used by `sted_fm` to download the datasets. Details of the helper scripts are in [`sted_fm/README.md`](sted_fm/README.md).


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from phenome import PhenoMe, load_dinov2_model, make_dataframe_metadata_fn
from phenome.utils import ModelWrapper

# Helper scripts live in Notebooks/examples/sted_fm, next to this notebook.
examples_dir = Path.cwd()
if not (examples_dir / "sted_fm").is_dir():
    examples_dir = examples_dir / "Notebooks" / "examples"
sys.path.insert(0, str(examples_dir.resolve()))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)


#### Choose dataset and model

**NAS:** 224×224 crops of STED images of PSD-95 under four experimental conditions: 0MgGlyBic, 48hTTX, Block, and GluGly.

**Optim:** 224×224 crops of STED images of four proteins: CaMKII, PSD-95, actin, and tubulin.

**Zooniverse:** crops of individual synaptic protein clusters. This is the only dataset with masks appropriate for extracting properties.


In [ ]:
# Choose one of 3 datasets:
DATASET_NAME = "NAS"
#DATASET_NAME = "Optim"
#DATASET_NAME = "Zooniverse"

# Choose one of 2 models from ("STED-FM", "DinoV2")
MODEL = "STED-FM"
#MODEL = "DinoV2"

#### Download and reshape the dataset

`DATASETS_PATH` is the local folder the archives are saved in. Change it if you want them somewhere else. The helpers in `sted_fm` write `images/`, `masks/` when the dataset has them, and `metadata.csv`.


In [ ]:
# Decide where to save datasets
DATASETS_PATH = "stedfm_datasets"
os.makedirs(DATASETS_PATH, exist_ok=True)

DATA_DIR = os.path.join(DATASETS_PATH, f"{DATASET_NAME}_dataset")
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
from sted_fm import download_dataset, transform_dataset

download_dataset(DATASET_NAME, DATA_DIR)
transform_dataset(DATASET_NAME, DATA_DIR, DATA_DIR)


### 1. Wrap the STED-FM pretrained model

In [ ]:
from stedfm import get_pretrained_model_v2


class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        model, cfg = get_pretrained_model_v2(
            name="mae-lightning-small",
            weights="MAE_SMALL_STED",
            in_channels=1,
            as_classifier=True,
        )
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        return self.model.forward_features(tensor)


if MODEL == "STED-FM":
    wrapper = STEDFMWrapper(device=device)
elif MODEL == "DinoV2":
    wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)


### 2. Use custom transforms

The default PhenoMe transforms apply standardisation based on ImageNet values, which isn't appropriate for the STED-FM model.

In [ ]:
import torchvision.transforms as T

TARGET_SIZE = 224

if MODEL == "STED-FM":
    # Transforms for STED-FM model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
    ])
elif MODEL == "DinoV2":
    # Transforms for DinoV2 model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])

### 3. Load data (find_files)

We assume CSV-based metadata. The STED datasets were already normalized. If raw data is used, it should be normalized based on percentile values from **entire images**, not by crop.

In [ ]:
import os

images_dir = os.path.join(DATA_DIR, "images")
masks_dir = os.path.join(DATA_DIR, "masks")  # Optional
metadata_dir = os.path.join(DATA_DIR, "metadata.csv")

metadata_df = pd.read_csv(metadata_dir)
metadata_fn = make_dataframe_metadata_fn(metadata_df)
file_df = pheno.find_files(images_dir, mask_dir=masks_dir, metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")


### 4. Process images

In [ ]:
# Base path to outputs: checkpoints and plots
OUTPUT_PATH = "output"
os.makedirs(OUTPUT_PATH, exist_ok=True)

CHECKPOINTS_PATH = os.path.join(OUTPUT_PATH, "checkpoints")
os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

In [ ]:
checkpoint_path = os.path.join(CHECKPOINTS_PATH, DATASET_NAME + f"_{MODEL}" + "_checkpoint.hdf5")

if MODEL == "STED-FM":
    pheno.process_images(wrapper, 
                        force_rgb=False,  # STED-FM model expects a single channel
                        checkpoint_path=checkpoint_path,
                        custom_transformations=custom_transforms,
                        )
elif MODEL == "DinoV2":
    pheno.process_images(wrapper, 
                    force_rgb=True,
                    channel_mode='combined',
                    checkpoint_path=checkpoint_path,
                    custom_transformations=custom_transforms,
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path=checkpoint_path)

### 5. Explore results

In [ ]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

### Generate plots

In [ ]:
PLOTS_PATH = os.path.join(OUTPUT_PATH, "plots")
os.makedirs(PLOTS_PATH, exist_ok=True)

if DATASET_NAME == "NAS":
    color_by = "condition"
    fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg", exclude={"condition":"Block"})
    fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_noblock_scatterplot.svg"))
elif DATASET_NAME == "Optim":
    color_by = "label"
elif DATASET_NAME == "Zooniverse":
    color_by = "protein"
    fig=pheno.plot_tsne(color_by="synapse_class", return_fig=True, render_mode="svg")
    fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_noblock_scatterplot.svg"))

fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg")
fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_scatterplot.svg"))